In [1]:
import functools
import getpass
import json
import os
from typing import List, Optional

import pandas as pd

# Settings
os.environ["QUERY_GEMINI"] = "True"  # if set to True, this generates a random result instead of prompting the LLM
os.environ["GEMINI_MODEL"] = "gemini-2.5-flash"  # the model to use on the Gemini API
os.environ["GEMINI_N"] = "1"  # the number of answers to generate per prompt
os.environ["GEMINI_TEMPERATURE"] = "1.0"  # the model temperature to use
os.environ["GEMINI_TIMEOUT"] = "120"  # the timeout for Gemini API calls
os.environ["GEMINI_REQUESTS_PER_MINUTE"] = "900"  # paid account request cap
os.environ["GEMINI_TOKENS_PER_MINUTE"] = "900000"  # paid account token cap
os.environ["GEMINI_RATE_LIMIT_BUFFER"] = "0.9"  # keep a safety buffer below the account caps
os.environ["PARALLEL_GEMINI_REQUESTS"] = "20"  # increase concurrency safely for paid limits
os.environ["TEMPLATE_DIR"] = "templates/"
os.environ["SQLITE_PATH"] = "gemini_results_1-N.sqlite3"  # the path to the SQLite database file. Set this to None to disable storage.

In [2]:
# IMPORTANT: Make sure you've run Cell 0 above first!
# Cell 0 sets the environment variables that fm_matcher reads when imported.
# If you skip Cell 0, the default values will be used instead of your settings.
from fm_matcher.utils_gemini import storage
from fm_matcher.utils_gemini.models import Attribute, Parameters, Relation, Result, Side
from fm_matcher.utils_gemini.prompt_building import PromptDesign, build_prompts
from fm_matcher.utils_gemini.prompt_sending import process_prompt_list, send_prompts
from fm_matcher.utils_gemini.prompt_postprocessing import postprocess_answers

# Only use 1-to-N mode
all_templates = ["oneToN"]
all_modes = [PromptDesign.oneToN]

/home/yuej18/miniconda3/envs/schema_eval/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/home/yuej18/schema_matching/fm_matcher/utils_gemini/prompt_sending.py:10: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  import google.generativeai as genai


In [3]:
benchmark = pd.read_csv("benchmark/CMS-OMOP-ground_truth.csv") #replace
for side in ("source", "target"):
    benchmark[[f"{side}_schema", f"{side}_relation", f"{side}_attribute"]] = benchmark[side].str.split(".", expand=True)
    benchmark[side] = benchmark[side].str.lower()
benchmark["benchmark"] = True

relation_combinations = benchmark[["source_relation", "target_relation"]].drop_duplicates().values

In [4]:
@functools.cache
def get_description(schema: str, table: str, attribute: Optional[str] = None) -> str:
    if attribute:
        filename = f"{schema}-{table}-{attribute}.txt"
        print(filename)
    else:
        filename = f"{schema}-table-{table}.txt"
        print(filename)
    filename_filter = lambda f: f.lower() == filename.lower()
    filename = next(filter(filename_filter, os.listdir("schema_documentations")))
    with open(os.path.join("schema_documentations", filename), "r") as desc_file:
        description = desc_file.read()
    return description


@functools.cache
def get_attributes(schema: str, table: str) -> List[str]:
    prefixes = [f"{schema}-{table}-", f"{schema}_{table}_"]
    suffix = ".txt"

    def table_filter(filename: str) -> bool:
        lower_filename = filename.lower()
        return any(
            lower_filename.startswith(prefix) and lower_filename.endswith(suffix)
            for prefix in prefixes
        )

    def extract_attr_name(filename: str) -> str:
        lower_filename = filename.lower()
        for prefix in prefixes:
            if lower_filename.startswith(prefix) and lower_filename.endswith(suffix):
                return filename[len(prefix) : -len(suffix)]
        return filename[:-len(suffix)]

    return [
        extract_attr_name(attr_file)
        for attr_file in filter(table_filter, os.listdir("schema_documentations/"))
    ]


@functools.cache
def get_relation(schema: str, table: str) -> Relation:
    schema, table = schema.lower(), table.lower()
    description = get_description(schema, table)
    attributes = [
        Attribute(
            name=attr_name.capitalize(),
            description=get_description(schema, table, attr_name),
        ) for attr_name in get_attributes(schema, table)
    ]

    side = Side.SOURCE if schema == "cms" else Side.TARGET #replace
    
    return Relation(
        name=table.capitalize(),
        side=side,
        description=description,
        attributes=attributes,
    )


parameters = [
    Parameters(
        source_relation=get_relation("cms", source), #replace
        target_relation=get_relation("omop", target),
    ) for source, target in relation_combinations
]
parameters = [storage.store_parameters(p) for p in parameters]

cms-table-beneficiarysummary.txt
cms-beneficiarysummary-bene_birth_dt.txt
cms-beneficiarysummary-bene_county_cd.txt
cms-beneficiarysummary-bene_death_dt.txt
cms-beneficiarysummary-bene_esrd_ind.txt
cms-beneficiarysummary-bene_hi_cvrage_tot_mons.txt
cms-beneficiarysummary-bene_hmo_cvrage_tot_mons.txt
cms-beneficiarysummary-bene_race_cd.txt
cms-beneficiarysummary-bene_sex_ident_cd.txt
cms-beneficiarysummary-bene_smi_cvrage_tot_mons.txt
cms-beneficiarysummary-benres_car.txt
cms-beneficiarysummary-benres_ip.txt
cms-beneficiarysummary-benres_op.txt
cms-beneficiarysummary-desynpuf_id.txt
cms-beneficiarysummary-medreimb_car.txt
cms-beneficiarysummary-medreimb_ip.txt
cms-beneficiarysummary-medreimb_op.txt
cms-beneficiarysummary-plan_cvrg_mos_num.txt
cms-beneficiarysummary-pppymt_car.txt
cms-beneficiarysummary-pppymt_ip.txt
cms-beneficiarysummary-pppymt_op.txt
cms-beneficiarysummary-sp_alzhdmta.txt
cms-beneficiarysummary-sp_chf.txt
cms-beneficiarysummary-sp_chrnkidn.txt
cms-beneficiarysummary-s

In [5]:
# Only define the oneToN template
templates = {
    "oneToN": [
        {
            "role": "user",
            "content": (
                "Act as a schema matcher for relational schemas. Your task is to create semantic matches"
                " that specify how the elements of the source schema and the target schema semantically"
                " correspond to one another. Two attributes semantically match if and only if there exists"
                " an invertible function that maps all values of one attribute to the other. First, I will"
                " input the name of an attribute from the source schema, a description of the attribute,"
                " the name of the relation in belongs to and a description of this relation. After that, I"
                " will input the same information of a single relation and all its attributes from the"
                " target schema."
            )
        },
        {
            "role": "user",
            "content": (
                "The attribute from the source schema is the following:\n"
                "Attribute name: '{{source_attribute.name}}'\n"
                "Attribute description: '{{source_attribute.description}}'\n"
                "Relation name: '{{source_relation.name}}'\n"
                "Relation description: '{{source_relation.description}}'"
            )
        },
        {
            "role": "user",
            "content": (
                "The relation from the target schema is the following:\n"
                "Relation name: '{{target_relation.name}}'\n"
                "Relation description: '{{target_relation.description}}'\n"
                "In the following, I will list all attributes of '{{target_relation.name}}'."
            )
        },
        {
            "role": "user",
            "content": (
                "Attribute name: '{{target_attribute.name}}'\n"
                "Attribute description: '{{target_attribute.description}}'"
            )
        },
        {
            "role": "user",
            "content": (
                "Explain which of the target attributes semantically match to '{{source_attribute.name}}'"
                " from '{{source_relation.name}}' of the source schema. Lets work this out step by step to"
                " make sure we get it correct. After your explanation, give a final decision"
                " JSON-formatted like this: `{ \"yes\": [], \"no\": [] }`. Under each of the following"
                " keys, list all target attributes of '{{target_relation.name}}' that apply: yes - if"
                " there is an invertible function that maps all values of the source attribute to the"
                " values of the target attribute; no - if there is no such function. Do not mention an"
                " attribute if there is not enough information to decide."
            )
        }
    ],
}

os.makedirs(os.getenv("TEMPLATE_DIR"), exist_ok=True)
for tpl_name in all_templates:
    with open(os.path.join(os.getenv("TEMPLATE_DIR"), f"{tpl_name}.json"), "w") as jsn:
        json.dump(templates[tpl_name.split("_")[-1]], jsn)

In [6]:
prompts = {
    parameter.meta["path"]: build_prompts(parameters=parameter, templates=all_templates, modes=all_modes)
    for parameter in parameters
}

prompts = {
    param_path: [storage.store_prompt(prompt) for prompt in prompt_list]
    for param_path, prompt_list in prompts.items()
}


# API Key Configuration
# Option 1: If GEMINI_API_KEY is already set as an environment variable, it will be used automatically
# Option 2: If not set, you'll be prompted to enter it (secure, won't display on screen)

if "GEMINI_API_KEY" not in os.environ or not os.environ.get("GEMINI_API_KEY"):
    os.environ["GEMINI_API_KEY"] = getpass.getpass("Your Gemini API key please:")

import google.generativeai as genai
genai.configure(api_key=os.environ["GEMINI_API_KEY"])

results = {}
start_idx = 0
total_parameters = len(parameters)

for i, parameter in enumerate(parameters[start_idx:], start=1):
    print(f"Processing parameter {i}/{total_parameters}...")
    answers = await process_prompt_list(parameter, prompts[parameter.meta["path"]])
    results[parameter.meta["path"]] = storage.store_result(postprocess_answers(parameter, answers))
    print(f"Finished parameter {i}/{total_parameters}")

Processing parameter 1/37...
Finished parameter 1/37
Processing parameter 2/37...
Finished parameter 2/37
Processing parameter 3/37...
Finished parameter 3/37
Processing parameter 4/37...
Finished parameter 4/37
Processing parameter 5/37...
Finished parameter 5/37
Processing parameter 6/37...
Finished parameter 6/37
Processing parameter 7/37...
Finished parameter 7/37
Processing parameter 8/37...
Finished parameter 8/37
Processing parameter 9/37...
Finished parameter 9/37
Processing parameter 10/37...
Finished parameter 10/37
Processing parameter 11/37...
Finished parameter 11/37
Processing parameter 12/37...
Finished parameter 12/37
Processing parameter 13/37...
Finished parameter 13/37
Processing parameter 14/37...
Finished parameter 14/37
Processing parameter 15/37...
Finished parameter 15/37
Processing parameter 16/37...
Finished parameter 16/37
Processing parameter 17/37...
Finished parameter 17/37
Processing parameter 18/37...
Finished parameter 18/37
Processing parameter 19/37..

In [7]:
def result_to_dataframe(result: Result) -> pd.DataFrame:
    df_data = []
    for attribute_pair, result_pair in result.pairs.items():
        for vote in result_pair.votes:
            left_scope = "1" if len(vote.answer.attributes.sources) == 1 else "n"
            right_scope = "1" if len(vote.answer.attributes.targets) == 1 else "n"
            task_scope = left_scope + "-to-" + right_scope
            source = "cms." + result.parameters.source_relation.name + "." + attribute_pair.source.name #replace
            target = "omop." + result.parameters.target_relation.name + "." + attribute_pair.target.name
            df_data.append({
                "task_scope": task_scope,
                "source": source.lower(),
                "source_attribute": attribute_pair.source.name,
                "source_relation": result.parameters.source_relation.name,
                "target": target.lower(),
                "target_attribute": attribute_pair.target.name,
                "target_relation": result.parameters.target_relation.name,
                "decision": str(vote.vote),
                "decision_index": vote.answer.index,
            })
    return pd.DataFrame(df_data)


result_df = pd.DataFrame()
for result in results.values():
    result_df = pd.concat((result_df, result_to_dataframe(result)))

result_df = result_df.merge(benchmark[["source", "target", "benchmark"]], on=["source", "target"], how="left").copy()
result_df["benchmark"] = result_df["benchmark"].fillna(False)
result_df.to_csv("results3/gemini_cms_results_1-N.csv", index=False) #replace
result_df

,task_scope,source,source_attribute,source_relation,target,target_attribute,target_relation,decision,decision_index,benchmark
0,1-to-n,cms.beneficiarysummary.bene_birth_dt,Bene_birth_dt,Beneficiarysummary,omop.person.birth_datetime,Birth_datetime,Person,yes,0,True
1,1-to-n,cms.beneficiarysummary.bene_birth_dt,Bene_birth_dt,Beneficiarysummary,omop.person.care_site_id,Care_site_id,Person,no,0,False
2,1-to-n,cms.beneficiarysummary.bene_birth_dt,Bene_birth_dt,Beneficiarysummary,omop.person.day_of_birth,Day_of_birth,Person,no,0,True
3,1-to-n,cms.beneficiarysummary.bene_birth_dt,Bene_birth_dt,Beneficiarysummary,omop.person.ethnicity_concept_id,Ethnicity_concept_id,Person,no,0,False
4,1-to-n,cms.beneficiarysummary.bene_birth_dt,Bene_birth_dt,Beneficiarysummary,omop.person.ethnicity_source_concept_id,Ethnicity_source_concept_id,Person,no,0,False
...,...,...,...,...,...,...,...,...,...,...
11542,1-to-n,cms.beneficiarysummary.sp_strketia,Sp_strketia,Beneficiarysummary,omop.visit_occurrence.visit_source_concept_id,Visit_source_concept_id,Visit_occurrence,no,0,False
11543,1-to-n,cms.beneficiarysummary.sp_strketia,Sp_strketia,Beneficiarysummary,omop.visit_occurrence.visit_source_value,Visit_source_value,Visit_occurrence,no,0,False
11544,1-to-n,cms.beneficiarysummary.sp_strketia,Sp_strketia,Beneficiarysummary,omop.visit_occurrence.visit_start_date,Visit_start_date,Visit_occurrence,no,0,False
11545,1-to-n,cms.beneficiarysummary.sp_strketia,Sp_strketia,Beneficiarysummary,omop.visit_occurrence.visit_start_datetime,Visit_start_datetime,Visit_occurrence,no,0,False
